# 从似然到置信区间

先从 Bernoulli 与正态模型构造估计量，再观察置信区间覆盖率和选择偏差。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(20260907)
np.set_printoptions(precision=5, suppress=True)

## 固定观测，改变参数
十次二元观测中七次成功。画出对数似然，比较内部最大点与全零/全一观测的边界最大点。似然曲线不是参数的概率密度。

In [ ]:
observed = np.array([1,1,0,1,1,0,1,0,1,1])
count, successes = len(observed), observed.sum()
p_hat = successes/count
p_grid = np.linspace(.001,.999,500)
log_likelihood = successes*np.log(p_grid)+(count-successes)*np.log1p(-p_grid)
fig, ax = plt.subplots()
ax.plot(p_grid, log_likelihood)
ax.axvline(p_hat, color='tab:red', label='sample mean')
ax.set(xlabel='p', ylabel='log likelihood'); ax.legend(); plt.show()
print('MLE:', p_hat)
for sample in [np.zeros(10), np.ones(10)]:
    print('边界样本 MLE:', sample.mean())
print('L(0.7)/L(0.5):', np.exp(successes*np.log(.7/.5)+(count-successes)*np.log(.3/.5)))

## 正态误差把似然最大化变成最小二乘
对三点回归，先求系数，再用残差平方和估计方差。MLE 的分母是 $n$，无偏估计的分母是 $n-d$。把系数沿某个方向扰动，观察在同一方差下对数似然怎样变化。

In [ ]:
design = np.array([[1.,0.],[1.,1.],[1.,2.]])
response = np.array([1.,2.,2.])
beta_ml = np.linalg.lstsq(design,response,rcond=None)[0]
errors = response-design@beta_ml
sse = errors@errors
n_obs, dimension = design.shape
variance_ml = sse/n_obs
variance_unbiased = sse/(n_obs-dimension)
print('系数、残差、SSE:', beta_ml,errors,sse)
print('方差 MLE 与无偏估计:', variance_ml,variance_unbiased)
for shift in [-.2,0.,.2]:
    candidate = beta_ml+np.array([0.,shift])
    difference = response-design@candidate
    log_value = -.5*n_obs*np.log(2*np.pi*variance_ml)-difference@difference/(2*variance_ml)
    print('斜率扰动、对数似然:',shift,log_value)

## 条件密度处理时间依赖
下面按正态 AR(1) 生成数据，给定首个观测后，对后续创新平方和求极小。该条件估计不包含初始分布的对数密度，也未施加平稳参数约束。

In [ ]:
series = np.empty(150);series[0]=0.
for t in range(1,len(series)):
    series[t]=.3+.7*series[t-1]+rng.normal(0,.5)
lag_design=np.column_stack([np.ones(len(series)-1),series[:-1]])
conditional_beta=np.linalg.lstsq(lag_design,series[1:],rcond=None)[0]
innovations=series[1:]-lag_design@conditional_beta
conditional_variance=np.mean(innovations**2)
print('条件估计 c、phi、v:',conditional_beta,conditional_variance)

已知总体标准差，固定第一项与选择最大项使用同样的区间公式。

In [ ]:
repeats=10000;n=25;sigma=2.;se=sigma/np.sqrt(n)
means=rng.normal(0,se,(repeats,20));chosen=means.max(axis=1)
print('coverage fixed, selected:',(abs(means[:,0])<=1.96*se).mean(),(abs(chosen)<=1.96*se).mean())
fig,axes=plt.subplots(1,2,figsize=(10,4),sharey=True)
for ax,values,title in zip(axes,[means[:40,0],chosen[:40]],['fixed first','selected maximum']):
    covered=abs(values)<=1.96*se
    for row,(value,hit) in enumerate(zip(values,covered)):
        ax.plot([value-1.96*se,value+1.96*se],[row,row],color='tab:blue' if hit else 'tab:red',alpha=.75)
    ax.axvline(0,color='black',linestyle='--')
    ax.set(title=title,xlabel='95% interval',ylabel='independent repetition')
plt.tight_layout();plt.show()

逐步计算 OLS 和 HC0 协方差：残差和设计共同进入中间矩阵。

In [ ]:
X=np.column_stack([np.ones(30),np.linspace(-1,1,30)]);y=X@np.array([1.,2.])+rng.normal(size=30)*(1+X[:,1])
beta=np.linalg.lstsq(X,y,rcond=None)[0];residual=y-X@beta
bread=np.linalg.inv(X.T@X);meat=X.T@(residual[:,None]**2*X);cov=bread@meat@bread
print('coefficients:',beta,'HC0 standard errors:',np.sqrt(np.diag(cov)))
print('orthogonality:',X.T@residual)

## 自己试一试

只比较固定第一项时为何覆盖接近 95%，选择最大项后为何下降？

## 反馈

前者符合原始重复抽样设计；后者条件于选择事件，普通区间没有调整搜索。

参数改变后应重新解释结果，不要求复现某次随机实验的小数。